# Klebsiella pneumoniae | MIL v5 - Multi-Head Attention + Focal Loss + LayerNorm

Mejoras sobre v4:
1. **Focal Loss** - penaliza errores dificiles, mejora recall clase R
2. **LayerNorm** sobre embeddings NTv3 antes de la atencion
3. **Multi-Head Attention MIL** (4 cabezas) - captura mecanismos paralelos
4. **Dropout mejorado** en embeddings + MIL head
5. **Ablacion Top-k** - experimento para paper (k=1,3,5,10 vs soft-attention)

> Este notebook NO reemplaza el entrenamiento v4; genera modelo independiente en modelo_MIL_v5_best_auc.pth

## Celda 0 - Anti-sueno y Drive

In [ ]:
try:
    from IPython.display import Javascript, display
    display(Javascript('''function ClickConnect(){document.querySelector("colab-toolbar-button#connect")?.click();document.querySelector("colab-dialog.yes-no-dialog paper-button#ok")?.click();}setInterval(ClickConnect,60000);'''))
    print("Anti-sueño activado")
    from google.colab import drive
    drive.mount("../data")
    IN_COLAB = True
    print("✅ Google Colab detectado y Drive montado.")
except Exception:
    IN_COLAB = False
    print("ℹ️ Entorno local detectado (no Google Colab). Omitiendo montaje de Drive.")


<IPython.core.display.Javascript object>

Anti-sueno activado
Drive already mounted at ../data; to attempt to forcibly remount, call drive.mount("../data", force_remount=True).


## Celda 1 - Instalar dependencias

In [ ]:
!pip install -q transformers captum biopython
!pip3 install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu121 --force-reinstall
print("Dependencias listas")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 126.7 MB/s eta 0:00:00
Looking in indexes: https://download.pytorch.org/whl/nightly/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.9/767.9 MB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 154.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 51.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 116.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 38.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 133.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 22.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56

Dependencias listas


## Celda 2 - Login HuggingFace

In [ ]:
import os
from huggingface_hub import login

# Autenticación con Hugging Face (necesario para descargar modelos)
# Utiliza la variable de entorno HF_TOKEN si está configurada, o solicita login interactivo
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
else:
    login()


## Celda 3 - Configuracion
> Ajusta aqui rutas e hiperparametros.

In [ ]:
import os

from pathlib import Path

# RUTAS
if 'IN_COLAB' in locals() and IN_COLAB:
    RUTA_BASE   = "../data"
else:
    REPO_ROOT = Path(os.getcwd()).resolve()
    if REPO_ROOT.name == 'notebooks':
        REPO_ROOT = REPO_ROOT.parent
    RUTA_BASE = str(REPO_ROOT)

def first_existing(paths, default):
    for p in paths:
        if os.path.exists(p):
            return p
    return default

DATASET_DIR = first_existing([
    f"{RUTA_BASE}/dataset_limpio",
    f"{RUTA_BASE}/data/dataset_limpio",
    f"{RUTA_BASE}/data",
], f"{RUTA_BASE}/data")

RUTA_TRAIN  = first_existing([f"{DATASET_DIR}/train_por_cepa.csv", f"{RUTA_BASE}/data/train_por_cepa.csv"], f"{DATASET_DIR}/train_por_cepa.csv")
RUTA_VAL    = first_existing([f"{DATASET_DIR}/val_por_cepa.csv", f"{RUTA_BASE}/data/val_por_cepa.csv"], f"{DATASET_DIR}/val_por_cepa.csv")
RUTA_MODELO = f"{RUTA_BASE}/modelo_MIL_v5.1_best_auc.pth"
RUTA_HIST   = f"{RUTA_BASE}/results/historial_entrenamiento_v5.1.csv" if not ('IN_COLAB' in locals() and IN_COLAB) else f"{RUTA_BASE}/historial_entrenamiento_v5.1.csv"

# MODELO
MODEL_NAME   = "InstaDeepAI/NTV3_100M_post"
SPECIES_ID   = 6
MAX_SEQ_LEN  = 512
HIDDEN_DIM   = 128
NUM_HEADS    = 4  # cabezas de atencion multi-head

# FOCAL LOSS
FOCAL_ALPHA  = 0.55  # peso clase R; >0.5 favorece recall R
FOCAL_GAMMA  = 2.0   # penalizacion ejemplos faciles

# ENTRENAMIENTO
LR            = 1e-5
WEIGHT_DECAY  = 0.01
MAX_EPOCHS    = 20
PATIENCE      = 3
WARMUP_FRAC   = 0.10

# BOLSAS
MAX_GENES     = 50
SUB_BATCH     = 64    # Aumentado para aprovechar mejor la VRAM de la GPU
EMBED_DROPOUT = 0.15  # dropout sobre embeddings NTv3

KEYWORDS_CARB = ["carbapenem", "beta-lactam", "OXA", "KPC", "NDM", "VIM", "IMP", "GES"]

print("Configuracion lista")
print(f"  Modelo        : {MODEL_NAME}")
print(f"  Multi-Head    : {NUM_HEADS} cabezas")
print(f"  Focal Loss    : alpha={FOCAL_ALPHA}, gamma={FOCAL_GAMMA}")
print(f"  Embed Dropout : {EMBED_DROPOUT}")
print(f"  Sub Batch     : {SUB_BATCH}")

# ===== FOLDS ST-BLOCKED (evaluacion solicitada por Hanqun) =====
# Los folds ST-blocked ya fueron generados externamente (algoritmo greedy multi-start)
# y exportados a Drive con esta convencion de nombres. Esta notebook NO regenera los folds,
# solo los consume para entrenar y evaluar.
DATASET_DIR = f"{RUTA_BASE}/dataset_limpio"
FOLD_IDS    = [1, 2, 3, 4, 5]
RUTA_TRAIN_FOLD = {k: f"{DATASET_DIR}/train_por_cepa_fold{k}.csv" for k in FOLD_IDS}
RUTA_VAL_FOLD   = {k: f"{DATASET_DIR}/val_por_cepa_fold{k}.csv"   for k in FOLD_IDS}

# Hiperparametros de entrenamiento para cada fold ST-blocked.
# Por defecto iguales a los del split aleatorio (no se cambia arquitectura ni regimen
# de entrenamiento); ajustar aqui si Colab necesita menos epocas por fold.
FOLD_MAX_EPOCHS = MAX_EPOCHS
FOLD_PATIENCE   = PATIENCE

RUTA_TABLA_RESUMEN = f"{RUTA_BASE}/comparativa_random_vs_st_blocked_resumen.csv"
RUTA_TABLA_FOLDS   = f"{RUTA_BASE}/comparativa_random_vs_st_blocked_detalle_folds.csv"

# Conteos de genomas esperados por fold (verificados externamente contra el manifest
# ST-blocked). Sirven de chequeo de seguridad: si no coinciden con lo que se carga en
# tiempo de ejecucion, la notebook se DETIENE en vez de seguir entrenando con datos
# incorrectos o caer de vuelta, en silencio, al split aleatorio.
EXPECTED_FOLD_COUNTS = {
    1: {"train": 3096, "val": 612},
    2: {"train": 3166, "val": 542},
    3: {"train": 3281, "val": 427},
    4: {"train": 2301, "val": 1407},
    5: {"train": 2988, "val": 720},
}

print(f"Folds ST-blocked configurados: {FOLD_IDS}")
for k in FOLD_IDS:
    print(f"  Fold {k}: train={RUTA_TRAIN_FOLD[k]} (esperado {EXPECTED_FOLD_COUNTS[k]['train']} cepas)")
    print(f"          val  ={RUTA_VAL_FOLD[k]} (esperado {EXPECTED_FOLD_COUNTS[k]['val']} cepas)")


Configuracion lista
  Modelo        : InstaDeepAI/NTV3_100M_post
  Multi-Head    : 4 cabezas
  Focal Loss    : alpha=0.55, gamma=2.0
  Embed Dropout : 0.15
  Sub Batch     : 64


## Celda 4 - Arquitectura NTv3MultiHeadMIL

Mejoras implementadas:
- **LayerNorm** sobre embeddings CLS antes de la atencion (mejora 2)
- **Dropout** sobre embeddings (mejora 4)
- **Multi-Head Attention MIL**: 4 redes de atencion paralelas, cada una puede capturar un mecanismo diferente (carbapenemasas, porinas, efflux, co-resistencias). Sus contextos se concatenan y proyectan a 768d antes de la clasificacion.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.optim import AdamW
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from tqdm.auto import tqdm
import gc

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"  GPU: {props.name} - VRAM: {props.total_memory/1024**3:.1f} GB")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
print("Tokenizador cargado")


class FocalLoss(nn.Module):
    """Focal Loss para clasificacion binaria con logits.
    Penaliza ejemplos facilmente clasificados, enfocando el aprendizaje
    en los casos dificiles de la clase R.
    alpha: peso para clase R. >0.5 incrementa recall R.
    gamma: exponente de modulacion. 0 = CrossEntropy estandar."""
    def __init__(self, alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction="none")
        pt = torch.exp(-ce)
        alpha_t = torch.where(targets == 1,
                               torch.tensor(self.alpha, device=logits.device),
                               torch.tensor(1 - self.alpha, device=logits.device))
        focal = alpha_t * (1 - pt) ** self.gamma * ce
        return focal.mean()


class NTv3MultiHeadMIL(nn.Module):
    """NTv3 (100M) + Multi-Head Attention MIL para clasificacion a nivel de aislado.

    Flujo:
      1. NTv3 genera embedding CLS [768] por gen.
      2. LayerNorm estabiliza escala (mejora 2).
      3. Dropout sobre embeddings (mejora 4).
      4. NUM_HEADS redes de atencion paralelas (mejora 3).
      5. Contextos concatenados -> proyeccion lineal -> [768] -> clasificador."""

    def __init__(self, model_name, hidden_dim=HIDDEN_DIM,
                 num_heads=NUM_HEADS, embed_dropout=EMBED_DROPOUT):
        super().__init__()
        self.ntv3 = AutoModel.from_pretrained(model_name, trust_remote_code=True)
        self.layer_norm   = nn.LayerNorm(768)
        self.embed_dropout = nn.Dropout(embed_dropout)
        self.attention_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(768, hidden_dim),
                nn.Tanh(),
                nn.Dropout(0.1),
                nn.Linear(hidden_dim, 1)
            ) for _ in range(num_heads)
        ])
        self.projection = nn.Linear(768 * num_heads, 768)
        self.proj_norm  = nn.LayerNorm(768)
        self.classifier = nn.Linear(768, 2)
        self.dropout    = nn.Dropout(0.1)

    def encode_genes(self, input_ids, attention_mask, species_ids):
        """NTv3 forward + LayerNorm + Dropout sobre embeddings CLS."""
        out = self.ntv3(
            input_ids=input_ids,
            attention_mask=attention_mask,
            species_ids=species_ids,
            output_hidden_states=True
        )
        emb = out.hidden_states[-1][:, 0, :]
        emb = self.layer_norm(emb)      # mejora 2
        emb = self.embed_dropout(emb)   # mejora 4
        return emb

    def classify(self, gene_embeddings):
        """Recibe [N_genes, 768]. Devuelve logits [1,2] y lista de pesos por cabeza."""
        head_contexts, attn_weights_per_head = [], []
        for attn_net in self.attention_heads:
            scores  = attn_net(gene_embeddings)         # [N, 1]
            weights = torch.softmax(scores, dim=0)      # [N, 1]
            context = (weights * gene_embeddings).sum(0)  # [768]
            head_contexts.append(context)
            attn_weights_per_head.append(weights)

        multi_context = torch.cat(head_contexts, dim=0)           # [768*num_heads]
        genome_emb    = self.proj_norm(self.projection(multi_context))  # [768]
        logits        = self.classifier(self.dropout(genome_emb.unsqueeze(0)))
        return logits, attn_weights_per_head


modelo = NTv3MultiHeadMIL(MODEL_NAME).to(device)
n_params = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
print(f"Modelo NTv3MultiHeadMIL instanciado")
print(f"  Parametros entrenables : {n_params/1e6:.1f}M")
print(f"  Cabezas de atencion    : {NUM_HEADS}")

Dispositivo: cuda
  GPU: NVIDIA L4 - VRAM: 22.0 GB


config.json: 0.00B [00:00, ?B/s]

configuration_ntv3_posttrained.py: 0.00B [00:00, ?B/s]

configuration_ntv3_pretrained.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/InstaDeepAI/ntv3_base_model:
- configuration_ntv3_pretrained.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/InstaDeepAI/ntv3_base_model:
- configuration_ntv3_posttrained.py
- configuration_ntv3_pretrained.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

modeling_ntv3_pretrained.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/InstaDeepAI/ntv3_base_model:
- modeling_ntv3_pretrained.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/InstaDeepAI/ntv3_base_model:
- modeling_ntv3_posttrained.py
- modeling_ntv3_pretrained.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/480M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/370 [00:00<?, ?it/s]

Modelo NTv3MultiHeadMIL instanciado
  Parametros entrenables : 122.8M
  Cabezas de atencion    : 4


## Celda 5 - Funciones Auxiliares de Entrenamiento y Evaluacion

In [ ]:
def cosine_diversity_loss(attn_list):
    """Calcula la similitud coseno entre las distribuciones de atencion de los distintos heads."""
    # attn_list tiene H tensores de [N, 1]
    attn = torch.cat(attn_list, dim=1).transpose(0, 1).unsqueeze(0)  # [1, H, N]
    _, H, _ = attn.shape
    loss = 0.0
    for i in range(H):
        for j in range(i + 1, H):
            sim = F.cosine_similarity(attn[:, i, :], attn[:, j, :], dim=-1).mean()
            loss += sim
    return loss


def encode_bag(model, input_ids, attention_mask, species_id_val, sub_batch=SUB_BATCH):
    """Procesa todos los genes de la bolsa por NTv3 en sub-batches."""
    embeddings = []
    n = input_ids.size(0)
    for s in range(0, n, sub_batch):
        e   = min(s + sub_batch, n)
        sp  = torch.full((e-s,), species_id_val, dtype=torch.long, device=device)
        emb = model.encode_genes(input_ids[s:e], attention_mask[s:e], sp)
        embeddings.append(emb)
    return torch.cat(embeddings, dim=0)


def evaluar_auc(model, loader):
    model.eval()
    y_true, y_score = [], []
    with torch.no_grad():
        for batch in loader:
            ids  = batch["input_ids"].squeeze(0).to(device).long()
            mask = batch["attention_mask"].squeeze(0).to(device).long()
            sp   = batch["species_ids"].item()
            with torch.amp.autocast('cuda'):
                embs = encode_bag(model, ids, mask, sp)
                logits, _ = model.classify(embs)
            prob = torch.softmax(logits, dim=1)[0, 1].item()
            y_true.append(batch["label"].item())
            y_score.append(prob)
    return roc_auc_score(y_true, y_score)


def entrenar_modelo(modelo, train_loader, val_loader, ruta_modelo, ruta_hist,
                     max_epochs=MAX_EPOCHS, patience=PATIENCE,
                     lr=LR, weight_decay=WEIGHT_DECAY, warmup_frac=WARMUP_FRAC,
                     tag="v5.1"):
    """Entrena con Focal Loss + Cosine Diversity Penalty + Early Stopping por AUC.
    Generaliza el loop de entrenamiento original (idéntico en logica y comportamiento)
    para poder reutilizarlo tanto en el split aleatorio como en cada fold ST-blocked."""
    optimizer = AdamW(modelo.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = FocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA)
    scaler    = torch.amp.GradScaler('cuda')

    total_steps  = len(train_loader) * max_epochs
    warmup_steps = int(total_steps * warmup_frac)
    scheduler    = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

    historial  = {"epoch": [], "train_loss": [], "val_auc": []}
    best_auc   = 0.0
    sin_mejora = 0

    print("\n" + "="*30)
    if "Fold" in str(tag):
        print(f"{str(tag).upper()}/5")
    else:
        print(f"{str(tag).upper()}")
    print("="*30 + "\n")

    for epoch in range(1, max_epochs + 1):
        modelo.train()
        total_loss = 0.0
        for batch in tqdm(train_loader, desc=f"Epoca {epoch}/{max_epochs}"):
            ids    = batch["input_ids"].squeeze(0).to(device).long()
            mask   = batch["attention_mask"].squeeze(0).to(device).long()
            sp_val = batch["species_ids"].item()
            label  = batch["label"].to(device)

            optimizer.zero_grad()

            with torch.amp.autocast('cuda'):
                embs              = encode_bag(modelo, ids, mask, sp_val)
                logits, attn_list = modelo.classify(embs)

                cls_loss = criterion(logits, label)
                div_loss = cosine_diversity_loss(attn_list)
                loss     = cls_loss + 0.01 * div_loss

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            total_loss += loss.item()
            torch.cuda.empty_cache()

        avg_loss = total_loss / len(train_loader)
        val_auc  = evaluar_auc(modelo, val_loader)
        historial["epoch"].append(epoch)
        historial["train_loss"].append(avg_loss)
        historial["val_auc"].append(val_auc)

        mejoro = val_auc > best_auc
        icono  = "NUEVO MEJOR" if mejoro else ""
        print(f"Epoca {epoch:2d} - Loss: {avg_loss:.4f} - Val AUC: {val_auc:.4f} {icono}")

        if mejoro:
            best_auc, sin_mejora = val_auc, 0
            torch.save({"epoch": epoch, "model_state_dict": modelo.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(), "auc": val_auc,
                        "config": {"max_genes": MAX_GENES, "sub_batch": SUB_BATCH,
                                    "lr": lr, "hidden_dim": HIDDEN_DIM,
                                    "num_heads": NUM_HEADS,
                                    "focal_alpha": FOCAL_ALPHA, "focal_gamma": FOCAL_GAMMA}},
                       ruta_modelo)
        else:
            sin_mejora += 1

        if sin_mejora >= patience:
            print(f"Early stopping - mejor AUC: {best_auc:.4f} (epoca {epoch-patience})")
            break

    pd.DataFrame(historial).to_csv(ruta_hist, index=False)
    print(f"Historial guardado en: {ruta_hist}")
    return historial, best_auc


if (not FORZAR_REENTRENAR_BASELINE) and os.path.exists(RUTA_MODELO):
    print(f"Checkpoint baseline (Random Split) ya existe en: {RUTA_MODELO}")
    print("No se reentrena el split aleatorio -> se evalua UNA SOLA VEZ como baseline, "
          "se reutiliza el AUC guardado. (FORZAR_REENTRENAR_BASELINE=True en la Celda 3 "
          "para forzar un reentrenamiento.)")
    ckpt_baseline = torch.load(RUTA_MODELO, map_location=device, weights_only=False)
    best_auc = ckpt_baseline["auc"]
    if os.path.exists(RUTA_HIST):
        historial = pd.read_csv(RUTA_HIST).to_dict(orient="list")
    else:
        historial = {"epoch": [], "train_loss": [], "val_auc": [best_auc]}
    print(f"AUC baseline (Random Split) cargado del checkpoint: {best_auc:.4f}")
else:
    historial, best_auc = entrenar_modelo(
        modelo, train_loader, val_loader, RUTA_MODELO, RUTA_HIST,
        tag="Random Split (baseline)")

import gc

# Liberar memoria del modelo entrenado con el split aleatorio antes de iterar los folds.
# Sus metricas y graficas ya se generaron en las celdas anteriores.
import gc
import matplotlib.pyplot as plt

--- {nombre} ---")
    print(f"  Train: {len(train_df_f):,} secuencias | {train_df_f['genome_id'].nunique()} cepas "
          f"| S={int((train_lbl==0).sum())} R={int((train_lbl==1).sum())}")
    print(f"  Val  : {len(val_df_f):,} secuencias | {val_df_f['genome_id'].nunique()} cepas "
          f"| S={int((val_lbl==0).sum())} R={int((val_lbl==1).sum())}")


def auditar_st_overlap(train_df_f, val_df_f, st_col):
    """Identifica las cepas de validacion cuyo ST nunca aparece en el train del fold.
    Devuelve (val_unseen_df, lista_genomas_no_vistos)."""
    st_train = train_df_f.drop_duplicates("genome_id").set_index("genome_id")[st_col]
    st_val   = val_df_f.drop_duplicates("genome_id").set_index("genome_id")[st_col]
    st_train_set   = set(st_train.dropna().unique())
    genomas_unseen = st_val[~st_val.isin(st_train_set)].index.tolist()
    val_unseen_df  = val_df_f[val_df_f["genome_id"].isin(genomas_unseen)].copy()
    n_val_genomas  = val_df_f["genome_id"].nunique()
    pct = 100 * len(genomas_unseen) / max(n_val_genomas, 1)
    print(f"  Auditoria ST: {len(genomas_unseen)}/{n_val_genomas} cepas de validacion "
          f"tienen un ST ausente en train ({pct:.1f}%)")
    return val_unseen_df, genomas_unseen


def validar_conteo_fold(fold_id, train_df_f, val_df_f):
    """Verifica, ANTES de entrenar, que el numero de cepas cargadas para este fold
    coincide con el conteo esperado (EXPECTED_FOLD_COUNTS). Si no coincide, detiene
    la ejecucion con un error explicito en vez de continuar silenciosamente o caer
    de vuelta a train_por_cepa.csv / val_por_cepa.csv."""
    esperado  = EXPECTED_FOLD_COUNTS[fold_id]
    obs_train = train_df_f["genome_id"].nunique()
    obs_val   = val_df_f["genome_id"].nunique()

    print(f"  Validacion de conteo - Fold {fold_id}:")
    print(f"    Esperado : train={esperado['train']} | val={esperado['val']}")
    print(f"    Observado: train={obs_train} | val={obs_val}")

    if obs_train != esperado["train"] or obs_val != esperado["val"]:
        raise RuntimeError(
            f"[ERROR] Fold {fold_id}: el conteo de cepas no coincide con lo esperado. "
            f"Esperado train={esperado['train']}/val={esperado['val']}, "
            f"observado train={obs_train}/val={obs_val}. Verifica que se este cargando "
            f"{RUTA_TRAIN_FOLD[fold_id]} y {RUTA_VAL_FOLD[fold_id]} y NO "
            f"train_por_cepa.csv/val_por_cepa.csv. Ejecucion detenida."
        )
    print(f"    OK: el Fold {fold_id} coincide con el conteo esperado.")


import matplotlib.pyplot as plt

def entrenar_y_evaluar_fold(fold_id):
    """Carga, entrena y evalua un fold ST-blocked completo. Devuelve un dict con metricas."""
    ruta_train_f = RUTA_TRAIN_FOLD[fold_id]
    ruta_val_f   = RUTA_VAL_FOLD[fold_id]

    print(f"\nCargando Fold {fold_id} (ST-blocked) desde: {ruta_train_f} | {ruta_val_f}")
    train_df_f = pd.read_csv(ruta_train_f)
    val_df_f   = pd.read_csv(ruta_val_f)

    # Chequeo de seguridad OBLIGATORIO antes de entrenar: si los conteos de cepas no
    # coinciden con lo esperado para este fold, se detiene la ejecucion. Esto evita
    # entrenar -en silencio- con el split aleatorio o con un fold incorrecto.
    validar_conteo_fold(fold_id, train_df_f, val_df_f)

    resumen_fold(f"Fold {fold_id} (ST-blocked)", train_df_f, val_df_f)

    st_col = detectar_columna_st(train_df_f)
    n_unseen, val_unseen_df = 0, None
    if st_col is None:
        print("  [AVISO] No se encontro una columna de Sequence Type (ST/MLST/'Sequence Type') "
              "en este fold. Para calcular unseen_st_auc se necesita una columna ST por genoma "
              "-tipicamente fusionada desde tus resultados de tipificacion (mlst/PubMLST) usando "
              "genome_id como llave- exportada tambien en estos CSV. Se omite unseen_st_auc para este fold.")
    else:
        val_unseen_df, genomas_unseen = auditar_st_overlap(train_df_f, val_df_f, st_col)
        n_unseen = len(genomas_unseen)

    _, train_loader_f = construir_loader(train_df_f, shuffle=True)
    _, val_loader_f    = construir_loader(val_df_f,   shuffle=False)

    modelo_fold = NTv3MultiHeadMIL(MODEL_NAME).to(device)
    ruta_ckpt_fold = f"{RUTA_BASE}/modelo_MIL_v5.1_fold{fold_id}_best_auc.pth"
    ruta_hist_fold = f"{RUTA_BASE}/historial_entrenamiento_v5.1_fold{fold_id}.csv"

    _, st_blocked_auc = entrenar_modelo(
        modelo_fold, train_loader_f, val_loader_f, ruta_ckpt_fold, ruta_hist_fold,
        max_epochs=FOLD_MAX_EPOCHS, patience=FOLD_PATIENCE,
        tag=f"Fold {fold_id}")

    # Recargar el mejor checkpoint del fold (no necesariamente la ultima epoca) para
    # la metrica de ST no visto, igual que se hace para el split aleatorio.
    # Generar grafica del fold
    historial_fold = pd.read_csv(ruta_hist_fold)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(historial_fold["epoch"], historial_fold["train_loss"], "o-", color="steelblue")
    ax1.set_title(f"Focal Loss por epoca (Fold {fold_id})")
    ax1.set_xlabel("Epoca"); ax1.set_ylabel("Focal Loss"); ax1.grid(alpha=0.3)
    ax2.plot(historial_fold["epoch"], historial_fold["val_auc"], "s-", color="darkgreen")
    best_ep = historial_fold["epoch"][historial_fold["val_auc"].idxmax()]
    ax2.axvline(x=best_ep, color="red", linestyle="--", alpha=0.7, label=f"Mejor epoca ({best_ep})")
    ax2.set_title(f"AUC en validacion por epoca (Fold {fold_id})")
    ax2.set_xlabel("Epoca"); ax2.set_ylabel("ROC AUC")
    ax2.legend(); ax2.grid(alpha=0.3)
    plt.suptitle(f"Historial de Entrenamiento - Fold {fold_id}", fontsize=13)
    plt.tight_layout()
    plt.savefig(f"{RUTA_BASE}/curva_entrenamiento_fold{fold_id}.png", dpi=120)
    plt.show()

    ckpt_fold = torch.load(ruta_ckpt_fold, map_location=device, weights_only=False)
    modelo_fold.load_state_dict(ckpt_fold["model_state_dict"])

    unseen_st_auc = None
    if st_col is not None:
        if n_unseen == 0:
            print("  Todas las cepas de validacion tienen un ST presente en train de este fold; "
                  "no hay subconjunto de ST no visto.")
        else:
            clases_presentes = val_unseen_df.drop_duplicates("genome_id")["label"].nunique()
            if clases_presentes < 2:
                print(f"  [AVISO] Las {n_unseen} cepas con ST no visto solo tienen una clase; "
                      "no se puede calcular ROC AUC en este subconjunto para este fold.")
            else:
                _, unseen_loader = construir_loader(val_unseen_df, shuffle=False)
                unseen_st_auc = evaluar_auc(modelo_fold, unseen_loader)
                print(f"  AUC en ST no visto ({n_unseen} cepas): {unseen_st_auc:.4f}")

    resultado = {
        "fold": fold_id,
        "st_column_detectada": st_col,
        "n_train_seqs": len(train_df_f), "n_val_seqs": len(val_df_f),
        "n_train_genomes": train_df_f["genome_id"].nunique(),
        "n_val_genomes": val_df_f["genome_id"].nunique(),
        "st_blocked_auc": st_blocked_auc,
        "unseen_st_auc": unseen_st_auc,
        "n_unseen_st_genomes": n_unseen,
    }

    del modelo_fold
    gc.collect()
    torch.cuda.empty_cache()
    return resultado


print("Funciones de evaluacion por fold listas.")

## Celda 6 - Entrenamiento Fold 1

In [ ]:
resultado_fold1 = entrenar_y_evaluar_fold(1)

## Celda 7 - Entrenamiento Fold 2

In [ ]:
resultado_fold2 = entrenar_y_evaluar_fold(2)

## Celda 8 - Entrenamiento Fold 3

In [ ]:
resultado_fold3 = entrenar_y_evaluar_fold(3)

## Celda 9 - Entrenamiento Fold 4

In [ ]:
resultado_fold4 = entrenar_y_evaluar_fold(4)

## Celda 10 - Entrenamiento Fold 5

In [ ]:
resultado_fold5 = entrenar_y_evaluar_fold(5)

## Celda 11 - Resumen Final 5-Fold

In [ ]:
import numpy as np
import pandas as pd

resultados_folds = [resultado_fold1, resultado_fold2, resultado_fold3, resultado_fold4, resultado_fold5]

st_blocked_aucs = np.array([r["st_blocked_auc"] for r in resultados_folds])
mean_st_blocked_auc = st_blocked_aucs.mean()
std_st_blocked_auc  = st_blocked_aucs.std(ddof=1)

unseen_vals = np.array([r["unseen_st_auc"] for r in resultados_folds if r["unseen_st_auc"] is not None])
mean_unseen_st_auc = unseen_vals.mean() if len(unseen_vals) > 0 else None
std_unseen_st_auc  = unseen_vals.std(ddof=1) if len(unseen_vals) > 1 else None

# --- Tabla detalle por fold ---
tabla_folds = pd.DataFrame([
    {
        "fold": r["fold"],
        "st_blocked_auc": r["st_blocked_auc"],
        "unseen_st_auc": r["unseen_st_auc"],
        "n_unseen_st_genomes": r["n_unseen_st_genomes"],
    }
    for r in resultados_folds
])
tabla_folds.to_csv(RUTA_TABLA_FOLDS, index=False)

print("\nTabla detalle por fold:")
print(tabla_folds.to_string(index=False))
print(f"\nTabla detalle guardada en: {RUTA_TABLA_FOLDS}")

# --- Tabla resumen ---
fila_resumen = {}
for r in resultados_folds:
    fila_resumen[f"Fold {r['fold']} ST-blocked AUC"] = r["st_blocked_auc"]
fila_resumen["Mean ST-blocked AUC"] = mean_st_blocked_auc
fila_resumen["Standard deviation ST-blocked AUC"] = std_st_blocked_auc

tabla_resumen = pd.DataFrame([fila_resumen])
tabla_resumen.to_csv(RUTA_TABLA_RESUMEN, index=False)

print("\n" + "="*70)
print("TABLA RESUMEN FINAL - 5-Fold Cross Validation")
print("="*70)
print(tabla_resumen.T.rename(columns={0: "Valor"}).to_string())
print("="*70)
print(f"Tabla resumen guardada en: {RUTA_TABLA_RESUMEN}")

print(f"\nMean +/- SD AUC ST-blocked (5 folds): {mean_st_blocked_auc:.4f} +/- {std_st_blocked_auc:.4f}")
if mean_unseen_st_auc is not None:
    sd_txt = f"{std_unseen_st_auc:.4f}" if std_unseen_st_auc is not None else "N/A (solo 1 fold con dato)"
    print(f"Mean +/- SD AUC ST no visto (folds con dato disponible): {mean_unseen_st_auc:.4f} +/- {sd_txt}")
else:
    print("AUC en ST no visto no disponible en ningun fold (columna ST no encontrada en los CSV exportados).")
